# ENEC 2026 · Day 2 · Governed RAG and Evaluation

**Time:** about 60 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Make retrieval respect document authority (APPROVED vs DRAFT vs SUPERSEDED), combine SQL with RAG, and measure the assistant with the 18-question evaluation set.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai 

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## 1. Build the index

`enec.RAGIndex` is the same chunk, embed and cosine search you wrote in the previous notebook, packaged with metadata filters.

In [ ]:
import pandas as pd, numpy as np, json, re
docs = enec.load_documents()
index = enec.RAGIndex.build(client, docs)
print(len(index.chunks), "chunks from", len(docs), "documents")
pd.DataFrame([{k: d[k] for k in ("doc_id", "status", "version", "date")} for d in docs])

## 2. The version trap

There are three versions of the inspection procedure: V1 (superseded), V2 (approved) and V3D (a newer draft). A naive system returns whatever is most similar. Watch the evidence.

In [ ]:
q = "Which inspection procedure should be followed for A-001?"
for h in index.search(q, k=6):
    print(f"{h['score']:.3f}  {h['doc_id']:<24} {h['status']:<11} v{h['version']}")

In [ ]:
# A naive prompt: no authority rules
hits = index.search(q, k=6)
naive = enec.ask(client, f"EVIDENCE:\n{enec.format_hits(hits)}\n\nQUESTION: {q}")
print("NAIVE:\n", naive)

In [ ]:
# Governed prompt: rules about status + citations
out = enec.rag_answer(client, index, q, k=6)
print("GOVERNED:\n", out["answer"])

In [ ]:
# Hard filter: retrieve only APPROVED documents
out = enec.rag_answer(client, index, q, k=4, statuses=["APPROVED"])
print("APPROVED ONLY:\n", out["answer"])
print([ (h["doc_id"], h["status"]) for h in out["hits"] ])

### Authority table

| Status | Meaning | Allowed use |
|---|---|---|
| APPROVED | Current, controlled document | Authority for guidance |
| DRAFT | Proposed, not yet approved | Mention as pending; never as authority |
| SUPERSEDED | Replaced by a later approved version | Historical context only |
| OPEN (work order) | A request for work | Evidence of intent, not authorisation |

**Which is better: filter in code, or instruct in the prompt?** Filtering is stricter. Instructing keeps useful context ("a draft exists and is not in force"). Many teams do both.

## 3. Hybrid question: SQL + RAG

*"What is the health score of A-001 and which procedure applies?"* needs a database fact **and** a document fact. RAG alone cannot answer it.

In [ ]:
con = enec.make_db()
facts = enec.run_sql(con, "SELECT asset_id, health_score, risk_level, recommended_action, open_work_orders, high_priority_open_work FROM asset_360 WHERE asset_id='A-001'")
doc_hits = index.search("Which inspection procedure applies to A-001 and what is its status?", k=4, statuses=["APPROVED"])

prompt = f"""DATABASE FACT (table asset_360):
{facts.to_csv(index=False)}

DOCUMENT EVIDENCE:
{enec.format_hits(doc_hits)}

QUESTION: What is the health score of A-001 and which procedure applies?
Label each part of your answer as DATABASE or DOCUMENT."""
print(enec.ask(client, prompt, system=enec.RAG_SYSTEM))

## 4. Evaluate: does retrieval find the right document?

`evaluation/rag_eval_questions.csv` has 18 questions with the expected source. First metric: **retrieval hit rate**, meaning the expected document appears in the top-k chunks. This costs only embeddings.

In [ ]:
ev = pd.read_csv(enec.EVAL_DIR / "rag_eval_questions.csv")
doc_ids = {d["doc_id"] for d in docs}

def expected_docs(s):
    return [p.strip() for p in str(s).split("+") if p.strip() in doc_ids]

def evaluate(k=4, statuses=None):
    rows = []
    for r in ev.itertuples():
        exp = expected_docs(r.expected_source)
        if not exp:
            continue                      # e.g. asset_360 only: that is a SQL question, not retrieval
        got = {h["doc_id"] for h in index.search(r.question, k=k, statuses=statuses)}
        rows.append({"question": r.question, "category": r.category, "expected": exp,
                     "hit": all(e in got for e in exp), "got": sorted(got)})
    return pd.DataFrame(rows)

res = evaluate(k=4)
print(f"Hit rate @4: {res.hit.mean():.0%}  ({int(res.hit.sum())}/{len(res)})")
res.groupby("category").hit.agg(["mean", "count"])

In [ ]:
res[~res.hit][["question", "expected", "got"]]      # failure analysis: read every miss

In [ ]:
# Does a larger k help? What does it cost (more text for the model to read)?
pd.DataFrame([{"k": k, "hit_rate": evaluate(k=k).hit.mean()} for k in (2, 4, 6, 8)])

## 5. Evaluate answers (optional LLM judge)

Retrieval can succeed and the answer can still be wrong. A second model can grade answers against the expected behaviour. Treat the judge as a **screening tool**, not ground truth: read the low scores yourself.

In [ ]:
RUN_JUDGE = True      # set False to skip the extra model calls
SAMPLE = 6

if RUN_JUDGE:
    judge_rows = []
    sample = ev[ev.expected_source.apply(lambda s: len(expected_docs(s)) > 0)].head(SAMPLE)
    for r in sample.itertuples():
        out = enec.rag_answer(client, index, r.question, k=4)
        jp = (f"QUESTION: {r.question}\nEXPECTED BEHAVIOUR: {r.expected_behavior}\nANSWER: {out['answer']}\n"
              'Score 1-5 how well the answer matches the expected behaviour and respects document authority. '
              'Return JSON: {"score": int, "reason": str}')
        j = json.loads(enec.ask(client, jp, json_mode=True))
        judge_rows.append({"question": r.question, "score": j.get("score"), "reason": j.get("reason")})
    pd.DataFrame(judge_rows)

## Takeaways

- **Authority is data, not a vibe.** Carry status and version with every chunk and use them.
- A hybrid question needs a hybrid system: SQL for facts, RAG for documents.
- **Measure.** A small, fixed evaluation set turns "it seems fine" into numbers you can improve.
- Read the failures. They tell you whether to change chunking, k, metadata or the prompt.

## Exercise

1. Add a question to the evaluation set that tests the DRAFT trap and check the hit rate.
2. Retrieval misses a question. Rewrite the query with an LLM (query rewriting), retrieve again, and compare.
3. Add a rule: if retrieved evidence contains both a DRAFT and an APPROVED version of the same procedure, the answer must say so explicitly. Test it.